In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# KONFIGURASI JENIS KACA
GLASS_TYPES = {
    0: {
        "name": "Building Windows Float Glass",
        "name_id": "Kaca Jendela Bangunan",
        "description": "Kaca jenis ini memiliki komposisi kimia yang sangat seimbang dengan kadar Magnesium (Mg) dan Kalsium (Ca) yang cukup tinggi, serta kadar Silikon (Si) yang dominan. Sifat kimiawi ini menghasilkan kaca dengan kejernihan optik yang sangat baik dan stabilitas struktural yang kuat. Karena karakteristiknya yang tahan terhadap tekanan lingkungan dan cuaca, kaca ini sangat ideal digunakan sebagai material utama untuk jendela, pintu, dan fasad bangunan komersial maupun residensial (float glass).",
        "icon": "🏢",
        "color": "#4facfe",
        "characteristics": {
            "Mg": "Tinggi (~3.4%)",
            "Ca": "Sedang (~8.5%)",
            "Ba": "Sangat Rendah (~0.01%)",
            "Si": "Tinggi (~72.8%)"
        }
    },
    1: {
        "name": "Headlamp / Tableware Glass",
        "name_id": "Kaca Lampu Depan / Peralatan Makan",
        "description": "Ciri khas utama dari kaca ini adalah tingginya kadar Barium (Ba) yang berfungsi untuk meningkatkan indeks bias dan kilau kaca, serta kadar Magnesium (Mg) yang sangat rendah. Penambahan Barium membuat kaca ini memiliki kemampuan memantulkan dan membiaskan cahaya dengan sangat baik. Oleh karena itu, kaca ini sangat cocok diaplikasikan pada lensa lampu depan kendaraan (headlamp) untuk memaksimalkan pencahayaan, serta digunakan pada peralatan makan dan gelas minum berkualitas tinggi yang mengutamakan estetika dan kilau.",
        "icon": "💡",
        "color": "#f093fb",
        "characteristics": {
            "Ba": "Tinggi (~1.08%)",
            "Mg": "Sangat Rendah (~0.45%)",
            "Al": "Sedang (~2.16%)",
            "Ca": "Sedang (~8.4%)"
        }
    },
    2: {
        "name": "Tableware / Container Glass",
        "name_id": "Kaca Peralatan Makan / Wadah",
        "description": "Kaca ini memiliki profil kimia yang unik dengan kadar Kalium (K) yang sangat tinggi dan hampir tidak mengandung Magnesium (Mg) sama sekali. Kandungan Kalium yang tinggi (sering dikombinasikan dengan Aluminium) secara signifikan meningkatkan ketahanan kimia dan ketahanan terhadap perubahan suhu ekstrem (thermal shock resistance). Sifat ini membuat kaca ini sangat aman, tahan lama, dan ideal untuk digunakan sebagai wadah penyimpanan, botol, serta peralatan makan dan minum (tableware) yang sering terpapar kondisi panas atau dingin.",
        "color": "#fa709a",
        "characteristics": {
            "K": "Sangat Tinggi (~6.21%)",
            "Mg": "Nol (0%)",
            "Al": "Tinggi (~3.03%)",
            "Si": "Rendah (~70.6%)"
        }
    },
    3: {
        "name": "Vehicle Windows Float Glass",
        "name_id": "Kaca Jendela Kendaraan",
        "description": "Kaca ini dirancang khusus untuk aplikasi otomotif, ditandai dengan kadar Kalsium (Ca) yang sangat tinggi dan nilai Indeks Bias (Refractive Index/RI) tertinggi di antara jenis kaca lainnya. Kalsium berperan penting dalam meningkatkan kekerasan (hardness) dan ketahanan fisik kaca terhadap goresan serta benturan, sementara kadar Besi (Fe) yang terkontrol memberikan sedikit tint alami. Kombinasi sifat mekanik yang kuat dengan kejernihan yang optimal menjadikan kaca ini standar industri untuk kaca jendela kendaraan yang mengutamakan keselamatan (safety) dan durabilitas.",
        "color": "#43e97b",
        "characteristics": {
            "Ca": "Tinggi (~11.0%)",
            "RI": "Tertinggi (~1.523)",
            "Mg": "Sedang (~2.05%)",
            "Fe": "Tertinggi (~0.09%)"
        }
    }
}

# 1. LOAD & EKSPLORASI DATA
print("1. LOAD & EKSPLORASI DATA")
df = pd.read_csv('glass.csv')

# Memisahkan fitur (X) dan label asli (y)
# Label asli TIDAK digunakan saat training, hanya untuk validasi di akhir
X = df.drop('label', axis=1)
y = df['label']

print(f"Shape dataset: {df.shape}")
print("\nStatistik Deskriptif Fitur (Perhatikan skala yang sangat berbeda):")
print(X.describe().round(2))


In [ ]:

# 2. STANDARDISASI FITUR
print("2. STANDARDISASI FITUR")
# K-Means berbasis jarak Euclidean. Fitur dengan skala besar (Si ~72) akan mendominasi 
# fitur berskala kecil (Ba ~0) jika tidak distandarisasi.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Standardisasi selesai. Mean ~0, Std ~1 untuk semua fitur.")

# 3 & 4. ELBOW METHOD & SILHOUETTE METHOD
print("3 & 4. EVALUASI: ELBOW & SILHOUETTE")
inertia = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    # random_state=42 untuk menjaga reproducibility (hasil selalu sama setiap kali dijalankan)
    # n_init=10 untuk menghindari warning dan memastikan konvergensi yang baik
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    inertia.append(kmeans.inertia_) # WCSS (Within-Cluster Sum of Square)
    sil_scores.append(silhouette_score(X_scaled, labels))

# Visualisasi Dual-Axis
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:blue'
ax1.set_xlabel('Jumlah Cluster (K)')
ax1.set_ylabel('Inertia (WCSS)', color=color)
ax1.plot(K_range, inertia, marker='o', color=color, label='Inertia (Elbow)')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(K_range, sil_scores, marker='s', color=color, label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Evaluasi K-Means: Elbow Method & Silhouette Score')
fig.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:

# 5. PILIH K OPTIMAL & JALANKAN K-MEANS FINAL
print("5. TRAINING MODEL FINAL")
# Berdasarkan plot di atas, Anda bisa memilih K. 
# Di sini kita pilih K=4 berdasarkan Elbow Method
optimal_k = 4
print(f"Melatih K-Means dengan K = {optimal_k}...")

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_scaled)

# Tambahkan nama jenis kaca berdasarkan cluster
df['glass_type'] = df['cluster'].map(lambda x: GLASS_TYPES[x]['name'])
df['glass_type_id'] = df['cluster'].map(lambda x: GLASS_TYPES[x]['name_id'])

print("\nDistribusi Cluster:")
print(df['cluster'].value_counts().sort_index())
print("\nDistribusi Jenis Kaca:")
print(df['glass_type'].value_counts())

# 6. VISUALISASI CLUSTER DENGAN PCA
print("6. VISUALISASI CLUSTER DENGAN PCA (Identifikasi Ambiguitas)")
# Reduksi 9 dimensi kimia menjadi 2 dimensi untuk visualisasi
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['cluster'] = df['cluster'].astype(str)
df_pca['label_asli'] = y
df_pca['glass_type'] = df['glass_type']

plt.figure(figsize=(12, 8))
# Trik: hue (warna) = Cluster K-Means, style (bentuk) = Label Asli
# Titik dengan warna dan bentuk yang berbeda adalah DATA AMBIGU/OVERLAP
scatter = sns.scatterplot(x='PC1', y='PC2', hue='cluster', style='label_asli', 
                palette=[GLASS_TYPES[i]['color'] for i in range(optimal_k)], 
                data=df_pca, alpha=0.7, s=100, edgecolor='black')

plt.title(f'Visualisasi Cluster (PCA) dengan K={optimal_k}\n(Bentuk = Label Asli | Warna = Cluster K-Means)\nTitik dengan bentuk/warna berbeda = Data Ambigu/Overlap')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varians)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varians)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Keterangan')
plt.tight_layout()
plt.show()

# 7. BANDINGKAN HASIL CLUSTER DENGAN LABEL ASLI
print("7. EVALUASI & INTERPRETASI")

# A. Evaluasi dengan Adjusted Rand Index (ARI)
ari = adjusted_rand_score(y, df['cluster'])
print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print("(Semakin mendekati 1, semakin mirip hasil cluster dengan jenis kaca asli)")

# B. Analisis Feature Importance (Cluster Centers)
# Mengembalikan nilai center dari skala standar ke skala asli agar bisa dibaca secara kimiawi
centers_original = scaler.inverse_transform(kmeans_final.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=X.columns)
centers_df['Cluster'] = [f"Cluster {i} - {GLASS_TYPES[i]['name_id']}" for i in range(optimal_k)]

print("\nRata-rata Komposisi Kimia per Cluster (Skala Asli):")
print(centers_df.round(3).to_string(index=False))




In [ ]:


# 8. IMPLEMENTASI DECISION TREE

print("8. IMPLEMENTASI DECISION TREE")

# Persiapan Data
# Kita menggunakan fitur asli (X) agar aturan pemisahan (split) di Decision Tree 
# mudah diinterpretasikan dalam satuan kimia asli (misal: Si > 72.5), 
# bukan dalam skala standar (z-score).
features = ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']
X_dt = df[features]
y_dt = df['cluster'] # Targetnya adalah label cluster hasil K-Means

# Split data menjadi 80% training dan 20% testing
# stratify=y_dt digunakan agar proporsi tiap cluster tetap seimbang di train dan test
X_train, X_test, y_train, y_test = train_test_split(
    X_dt, y_dt, test_size=0.2, random_state=42, stratify=y_dt
)

print(f"Data Training: {X_train.shape[0]} sampel")
print(f"Data Testing: {X_test.shape[0]} sampel\n")

# Inisialisasi dan Training Model
# max_depth=4 dibatasi agar pohon tidak terlalu dalam (overfitting) dan mudah divisualisasikan
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42, criterion='gini')
dt_model.fit(X_train, y_train)

# Evaluasi Model
y_pred = dt_model.predict(X_test)

print("--- HASIL EVALUASI DECISION TREE ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")

# Tambahkan parameter 'labels' agar mapping dengan 'target_names' benar
print(classification_report(y_test, y_pred, 
                            labels=range(optimal_k), # Menentukan urutan label (0, 1, 2, 3)
                            target_names=[f"Cluster {i} - {GLASS_TYPES[i]['name_id']}" for i in range(optimal_k)],
                            zero_division=0)) # Menghindari warning jika ada cluster yang tidak ada di test set


# Visualisasi Pohon Keputusan (Decision Tree)
plt.figure(figsize=(20, 12))
plot_tree(dt_model, 
          feature_names=features, 
          class_names=[f'Cluster {i} - {GLASS_TYPES[i]["name_id"]}' for i in range(optimal_k)], 
          filled=True, 
          rounded=True, 
          fontsize=10)
plt.title('Visualisasi Decision Tree untuk Memprediksi Cluster K-Means')
plt.tight_layout()
plt.show()

# Feature Importance (Seberapa penting setiap unsur kimia dalam membedakan cluster)
importances = dt_model.feature_importances_
feature_imp_df = pd.DataFrame({
    'Fitur': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Fitur', data=feature_imp_df, palette='viridis')
plt.title('Tingkat Kepentingan Fitur (Feature Importance) pada Decision Tree')
plt.xlabel('Nilai Importance')
plt.ylabel('Unsur Kimia')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:

# 9. MENYIMPAN MODEL KE FILE (.pkl)
print("9. MENYIMPAN MODEL KE FILE (PICKLE/JOBLIB)")

joblib.dump(scaler, 'scaler.pkl')
print(" Scaler berhasil disimpan sebagai 'scaler.pkl'")
joblib.dump(kmeans_final, 'kmeans_model.pkl')
print(" Model K-Means berhasil disimpan sebagai 'kmeans_model.pkl'")
joblib.dump(dt_model, 'decision_tree_model.pkl')
print(" Model Decision Tree berhasil disimpan sebagai 'decision_tree_model.pkl'")
feature_names = X.columns.tolist()
joblib.dump(feature_names, 'feature_names.pkl')
print(" Daftar fitur berhasil disimpan sebagai 'feature_names.pkl'")
joblib.dump(GLASS_TYPES, 'glass_types.pkl')
print(" Informasi jenis kaca berhasil disimpan sebagai 'glass_types.pkl'")

# 10. GABUNGKAN SEMUA MODEL KE DALAM SATU PIPELINE
print("10. MENGGABUNGKAN MODEL KE DALAM SATU PIPELINE")


# Definisikan Class Pipeline Kustom
class GlassAnalysisPipeline:
    def __init__(self, scaler, kmeans, decision_tree, feature_names, glass_types):
        self.scaler = scaler
        self.kmeans = kmeans
        self.decision_tree = decision_tree
        self.feature_names = feature_names
        self.glass_types = glass_types

    def predict(self, input_data_dict):
        """
        Menerima data baru dalam bentuk dictionary, 
        lalu mengembalikan prediksi dari K-Means dan Decision Tree beserta informasi jenis kaca.
        """
        # 1. Ubah input dictionary menjadi DataFrame dengan urutan kolom yang benar
        df_input = pd.DataFrame([input_data_dict], columns=self.feature_names)
        # 2. Lakukan scaling pada data (wajib untuk K-Means)
        scaled_data = self.scaler.transform(df_input)
        # 3. Prediksi menggunakan K-Means
        kmeans_cluster = self.kmeans.predict(scaled_data)[0]
        # 4. Prediksi menggunakan Decision Tree (menggunakan data asli, bukan scaled)
        dt_cluster = self.decision_tree.predict(df_input)[0]
        # 5. Dapatkan informasi jenis kaca
        kmeans_glass_info = self.glass_types[int(kmeans_cluster)]
        dt_glass_info = self.glass_types[int(dt_cluster)]
        # 6. Kembalikan hasil dalam bentuk dictionary
        return {
            "kmeans_cluster": int(kmeans_cluster),
            "kmeans_glass_type": kmeans_glass_info['name'],
            "kmeans_glass_type_id": kmeans_glass_info['name_id'],
            "kmeans_description": kmeans_glass_info['description'],
            "decision_tree_cluster": int(dt_cluster),
            "decision_tree_glass_type": dt_glass_info['name'],
            "decision_tree_glass_type_id": dt_glass_info['name_id'],
            "decision_tree_description": dt_glass_info['description'],
            "message": f"Data diprediksi masuk ke Cluster {int(kmeans_cluster)} ({kmeans_glass_info['name_id']}) berdasarkan K-Means dan Cluster {int(dt_cluster)} ({dt_glass_info['name_id']}) berdasarkan Decision Tree."
        }

# Muat semua file .pkl yang terpisah
print("Memuat model-model terpisah...")
scaler = joblib.load('scaler.pkl')
kmeans = joblib.load('kmeans_model.pkl')
dt = joblib.load('decision_tree_model.pkl')
features = joblib.load('feature_names.pkl')
glass_types = joblib.load('glass_types.pkl')

# Buat instance dari Class Pipeline kita
print("Menggabungkan ke dalam satu objek Pipeline...")
full_pipeline = GlassAnalysisPipeline(
    scaler=scaler,
    kmeans=kmeans,
    decision_tree=dt,
    feature_names=features,
    glass_types=glass_types
)

# Simpan instance tersebut menjadi SATU file .pkl
output_filename = 'glass_full_pipeline.pkl'
joblib.dump(full_pipeline, output_filename)

print(f"\n Sukses! Semua model telah digabung dan disimpan sebagai '{output_filename}'")
print("Anda sekarang bisa menghapus file .pkl yang terpisah jika tidak diperlukan lagi.")


In [ ]:

# 11. TEST PIPELINE
print("11. TEST PIPELINE DENGAN DATA CONTOH")

# Contoh data uji
test_data = {
    'RI': 1.52,
    'Na': 13.5,
    'Mg': 2.5,
    'Al': 1.4,
    'Si': 72.5,
    'K': 0.5,
    'Ca': 8.5,
    'Ba': 0.1,
    'Fe': 0.05
}

print("\nData Uji:")
for key, value in test_data.items():
    print(f"  {key}: {value}")

# Prediksi menggunakan pipeline
result = full_pipeline.predict(test_data)

print("\nHasil Prediksi:")
print(f"  K-Means Cluster: {result['kmeans_cluster']}")
print(f"  K-Means Jenis Kaca: {result['kmeans_glass_type']}")
print(f"  K-Means Deskripsi: {result['kmeans_description']}")
print(f"\n  Decision Tree Cluster: {result['decision_tree_cluster']}")
print(f"  Decision Tree Jenis Kaca: {result['decision_tree_glass_type']}")
print(f"  Decision Tree Deskripsi: {result['decision_tree_description']}")
print(f"\n  Pesan: {result['message']}")

print("PROSES SELESAI!")
print("\nFile yang dihasilkan:")
print("  - glass_full_pipeline.pkl (Pipeline lengkap)")
print("  - scaler.pkl")
print("  - kmeans_model.pkl")
print("  - decision_tree_model.pkl")
print("  - feature_names.pkl")
print("  - glass_types.pkl")